[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_12/MFM_ch12_inference/MFM_ch12_inference.ipynb)

# MFM_ch12_inference

Inference on option-implied quantities: no-arbitrage checks of the Bitcoin SVI fits (Lee wing bound, Gatheral-Jacquier butterfly function, calendar condition); truncation and discretisation of the Bitcoin VIX and the jump bias of the log contract in a Merton model; the VIX as a forecast (joint Mincer-Zarnowitz test, logs, instrumental variables, encompassing HAR-RV, out-of-sample QLIKE and Diebold-Mariano); ex-ante variance risk premium and 1-12-month S&P 500 excess returns with Newey-West, Hansen-Hodrick and Hodrick (1992) standard errors and a bootstrap under the null.

*Modelling Financial Markets (MFM), Chapter 12: Options and the Volatility Surface. Author: Daniel Traian Pele.*

In [ ]:
import os
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy import stats, optimize, integrate
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

N, n = stats.norm.cdf, stats.norm.pdf

## Style and helpers

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'   # doar linii de referinta, benzi, grila
LightGray = '#DADADA'
Teal     = '#17A2B8'

HERE = '.'
SEED = 42
BASE = {'S': 100.0, 'K': 100.0, 'T': 0.25, 'r': 0.04, 'sigma': 0.2}
HEDGE = {'S0': 100.0, 'K': 100.0, 'T': 0.25, 'r': 0.04, 'sigma': 0.2, 'mu': 0.08, 'nstep': 252, 'npath': 20000}


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def fig_legend_bottom(fig, handles, ncol=3, y=0.0):
    """O singura legenda pentru o figura cu mai multe panouri, sub figura."""
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def nw_mean(x, lags):
    """Media si eroarea standard Newey-West (HAC) a unei serii."""
    x = np.asarray(x, float)
    m = sm.OLS(x, np.ones_like(x)).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    return float(m.params[0]), float(m.bse[0])


def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, np.ndarray)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, pd.Timestamp):
        return str(o)
    return o

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# copia locala a folderului data/market, daca notebook-ul ruleaza din repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
SNAP_DIR = next((d for d in ('.', '../Quantlets/Ch_12', '../../Quantlets/Ch_12')
                 if os.path.exists(os.path.join(d, 'ch12_deribit_snapshot.csv'))), '.')
END = '2026-09-18'

# nume -> (simbol, coloana, tip)
SERIES = {
    'sp500': ('GSPC.INDX', 'close', 'index'),
    'spy':   ('SPY.US', 'adjusted_close', 'etf'),
    'btc':   ('BTC-USD.CC', 'close', 'crypto'),
    'vix':   ('VIX.INDX', 'close', 'vol'),
    'vix9d': ('VIX9D.INDX', 'close', 'vol'),
    'vix3m': ('VIX3M.INDX', 'close', 'vol'),
    'vvix':  ('VVIX.INDX', 'close', 'vol'),
}
DERIBIT = 'https://www.deribit.com/api/v2/public/'
SNAP_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/Quantlets/Ch_12/ch12_deribit_snapshot.csv'


def read_market(symbol):
    """Citeste data/market/<SIMBOL>.csv local sau din repo-ul GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def load_close(name, start=None, end=END):
    """Pretul (sau nivelul indicelui) zilnic; indicii: doar zilele lucratoare, fara zilele cu valoare neschimbata."""
    symbol, col, kind = SERIES[name]
    s = read_market(symbol)[col].loc[start:end]
    s = s[s > 0].dropna()
    if kind != 'crypto':
        s = s[s.index.dayofweek < 5]
    if kind == 'index':
        s = s[s.diff() != 0]
    return s.rename(name)


def log_returns(name, start=None, end=END):
    """Randamente log pe calendarul propriu al seriei."""
    return np.log(load_close(name, start, end)).diff().dropna().rename(name)


def spy_open_close(start=None, end=END):
    """Deschiderea si inchiderea zilnica SPY (preturi neajustate: raportul din aceeasi zi nu depinde de ajustari)."""
    d = read_market('SPY.US').loc[start:end, ['open', 'close']]
    d = d[(d > 0).all(axis=1)]
    return d[d.index.dayofweek < 5]


def _get(method, **params):
    url = DERIBIT + method + '?' + '&'.join(f'{k}={v}' for k, v in params.items())
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    return json.load(urllib.request.urlopen(req, timeout=60))['result']


def deribit_chain(path=None, refresh=False):
    """Lantul de optiuni BTC (preturi de marcaj, volatilitati implicite) la momentul instantaneului."""
    path = path or os.path.join(SNAP_DIR, 'ch12_deribit_snapshot.csv')
    if not refresh:
        for src in (path, SNAP_RAW):
            try:
                return pd.read_csv(src, parse_dates=['snapshot_utc', 'expiry'])
            except Exception:
                pass
    rows = _get('get_book_summary_by_currency', currency='BTC', kind='option')
    t0 = pd.Timestamp.now(tz='UTC').floor('min').tz_localize(None)
    out = []
    for x in rows:
        _, exp, strike, cp = x['instrument_name'].split('-')
        out.append(dict(snapshot_utc=t0, instrument=x['instrument_name'],
                        expiry=pd.to_datetime(exp, format='%d%b%y') + pd.Timedelta(hours=8),
                        strike=float(strike), type='call' if cp == 'C' else 'put',
                        mark_iv=x['mark_iv'], mark_price=x['mark_price'], bid=x['bid_price'], ask=x['ask_price'],
                        forward=x['underlying_price'], index=x['estimated_delivery_price'],
                        open_interest=x['open_interest'], volume_usd=x['volume_usd']))
    d = pd.DataFrame(out).sort_values(['expiry', 'strike', 'type']).reset_index(drop=True)
    d.to_csv(path, index=False)
    return d


def dvol_history(start='2021-03-24', end=END):
    """Indicele DVOL (volatilitatea implicita pe 30 de zile a optiunilor BTC, anualizata, in %), valori zilnice."""
    t1 = int(pd.Timestamp(end).tz_localize('UTC').timestamp() * 1000) + 86_400_000
    t0 = int(pd.Timestamp(start).tz_localize('UTC').timestamp() * 1000)
    data = []
    while True:
        r = _get('get_volatility_index_data', currency='BTC', start_timestamp=t0, end_timestamp=t1, resolution='1D')
        data += r['data']
        if not r.get('continuation') or r['continuation'] <= t0:
            break
        t1 = r['continuation']
    d = pd.DataFrame(data, columns=['t', 'open', 'high', 'low', 'close']).drop_duplicates('t')
    d.index = pd.to_datetime(d['t'], unit='ms').dt.normalize()
    return d.sort_index()['close'].loc[start:end].rename('dvol')

## Option pricing tools

In [ ]:
def bs_price(S, K, T, r, sigma, kind='call', q=0.0):
    """Pretul Black-Scholes al unei optiuni europene."""
    S, K, T, sigma = map(np.asarray, (S, K, T, sigma))
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if kind == 'call':
        return S * np.exp(-q * T) * N(d1) - K * np.exp(-r * T) * N(d2)
    return K * np.exp(-r * T) * N(-d2) - S * np.exp(-q * T) * N(-d1)


def bs_greeks(S, K, T, r, sigma, kind='call', q=0.0):
    """Delta, gamma, vega (la 1 punct de volatilitate), theta (pe zi calendaristica), rho; plus d1, d2."""
    S, K, T, sigma = map(np.asarray, (S, K, T, sigma))
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    gamma = np.exp(-q * T) * n(d1) / (S * sigma * np.sqrt(T))
    vega = S * np.exp(-q * T) * n(d1) * np.sqrt(T) / 100
    common = -S * np.exp(-q * T) * n(d1) * sigma / (2 * np.sqrt(T))
    if kind == 'call':
        delta = np.exp(-q * T) * N(d1)
        theta = common - r * K * np.exp(-r * T) * N(d2) + q * S * np.exp(-q * T) * N(d1)
        rho = K * T * np.exp(-r * T) * N(d2) / 100
    else:
        delta = -np.exp(-q * T) * N(-d1)
        theta = common + r * K * np.exp(-r * T) * N(-d2) - q * S * np.exp(-q * T) * N(-d1)
        rho = -K * T * np.exp(-r * T) * N(-d2) / 100
    return dict(d1=d1, d2=d2, delta=delta, gamma=gamma, vega=vega, theta=theta / 365, rho=rho)


def implied_vol(price, S, K, T, r, kind='call', q=0.0, lo=1e-4, hi=5.0):
    """Volatilitatea implicita: solutia unica a ecuatiei BS(sigma) = pret (vega > 0)."""
    f = lambda s: bs_price(S, K, T, r, s, kind, q) - price
    if f(lo) > 0 or f(hi) < 0:
        return np.nan
    return optimize.brentq(f, lo, hi, xtol=1e-10)


def newton_iv(price, S, K, T, r, sigma0=0.2, kind='call', steps=4):
    """Pasii metodei Newton: sigma_{k+1} = sigma_k - (BS(sigma_k) - pret) / vega(sigma_k)."""
    out, s = [], sigma0
    for _ in range(steps):
        p = float(bs_price(S, K, T, r, s, kind))
        v = float(bs_greeks(S, K, T, r, s, kind)['vega']) * 100
        s_new = s - (p - price) / v
        out.append(dict(sigma=s, price=p, vega=v, sigma_next=s_new))
        s = s_new
    return out


def crr_price(S, K, T, r, sigma, steps, kind='call', american=False):
    """Arborele binomial Cox-Ross-Rubinstein: u = exp(sigma sqrt(dt)), d = 1/u, p = (e^{r dt} - d)/(u - d)."""
    dt = T / steps
    u = np.exp(sigma * np.sqrt(dt)); d = 1 / u
    p = (np.exp(r * dt) - d) / (u - d)
    disc = np.exp(-r * dt)
    j = np.arange(steps + 1)
    ST = S * u ** (steps - j) * d ** j
    V = np.maximum(ST - K, 0) if kind == 'call' else np.maximum(K - ST, 0)
    for i in range(steps - 1, -1, -1):
        V = disc * (p * V[:-1] + (1 - p) * V[1:])
        if american:
            Si = S * u ** (i - np.arange(i + 1)) * d ** np.arange(i + 1)
            V = np.maximum(V, (Si - K) if kind == 'call' else (K - Si))
    return float(V[0])


def merton_price(S, K, T, r, sigma, lam, mu_j, sig_j, kind='call', nmax=60):
    """Merton (1976): salturi log-normale ln(1+J) ~ N(mu_j, sig_j^2), intensitate lam; serie Poisson de preturi BS."""
    kappa = np.exp(mu_j + 0.5 * sig_j ** 2) - 1
    lam2 = lam * (1 + kappa)
    tot = 0.0
    for k in range(nmax):
        s_k = np.sqrt(sigma ** 2 + k * sig_j ** 2 / T)
        r_k = r - lam * kappa + k * np.log(1 + kappa) / T
        w = stats.poisson.pmf(k, lam2 * T)
        tot = tot + w * bs_price(S, K, T, r_k, s_k, kind)
    return tot


def heston_price(S, K, T, r, v0, kappa, theta, xi, rho, kind='call'):
    """Heston (1993): pretul call prin formula Lewis (2001), cu functia caracteristica a lui ln(S_T/F) (forma stabila)."""
    def phi(u):
        d = np.sqrt((rho * xi * 1j * u - kappa) ** 2 + xi ** 2 * (1j * u + u ** 2))
        g = (kappa - rho * xi * 1j * u - d) / (kappa - rho * xi * 1j * u + d)
        e = np.exp(-d * T)
        C = kappa * theta / xi ** 2 * ((kappa - rho * xi * 1j * u - d) * T - 2 * np.log((1 - g * e) / (1 - g)))
        D = (kappa - rho * xi * 1j * u - d) / xi ** 2 * (1 - e) / (1 - g * e)
        return np.exp(C + D * v0)
    x = np.log(S / K) + r * T
    f = lambda u: (np.exp(1j * u * x) * phi(u - 0.5j)).real / (u ** 2 + 0.25)
    call = S - np.sqrt(S * K) * np.exp(-r * T / 2) / np.pi * integrate.quad(f, 0, 200, limit=500, epsabs=1e-12)[0]
    return call if kind == 'call' else call - S + K * np.exp(-r * T)


def svi_w(k, a, b, rho, m, s):
    """SVI (Gatheral): varianta totala implicita w(k) = a + b (rho (k - m) + sqrt((k - m)^2 + s^2)), k = ln(K/F)."""
    return a + b * (rho * (k - m) + np.sqrt((k - m) ** 2 + s ** 2))


def svi_fit(k, w, weights=None):
    """Estimarea SVI prin cele mai mici patrate (ponderate), cu restrictii b >= 0, |rho| < 1, s > 0."""
    k, w = np.asarray(k), np.asarray(w)
    wt = np.ones_like(w) if weights is None else np.asarray(weights)
    def loss(p):
        return np.sum(wt * (svi_w(k, *p) - w) ** 2)
    best = None
    for rho0 in (-0.5, 0.0, 0.3):
        for s0 in (0.05, 0.2):
            x0 = [max(w.min() * 0.5, 1e-4), 0.1, rho0, 0.0, s0]
            res = optimize.minimize(loss, x0, method='L-BFGS-B',
                                    bounds=[(-1, 5), (1e-6, 10), (-0.999, 0.999), (-2, 2), (1e-4, 3)])
            if best is None or res.fun < best.fun:
                best = res
    a, b, rho, m, s = best.x
    return dict(a=a, b=b, rho=rho, m=m, s=s, rmse=np.sqrt(best.fun / wt.sum()),
                wmin=a + b * s * np.sqrt(1 - rho ** 2))


def variance_from_strip(K, Q, F, T, r=0.0):
    """Varianta implicita (formula VIX): 2/T sum dK/K^2 e^{rT} Q(K) - 1/T (F/K0 - 1)^2, Q = pretul OTM (mid)."""
    K, Q = np.asarray(K, float), np.asarray(Q, float)
    o = np.argsort(K); K, Q = K[o], Q[o]
    dK = np.empty_like(K)
    dK[1:-1] = (K[2:] - K[:-2]) / 2
    dK[0], dK[-1] = K[1] - K[0], K[-1] - K[-2]
    K0 = K[K <= F].max()
    return 2 / T * np.sum(dK / K ** 2 * np.exp(r * T) * Q) - (F / K0 - 1) ** 2 / T


def delta_hedge(paths, K, T, r, sigma_imp, n_rebal, kind='call'):
    """Vinde o optiune la pretul BS(sigma_imp) si acopera delta de n_rebal ori pana la scadenta.
    paths: matrice (n_traiectorii, n_pasi+1) de preturi pe o grila uniforma; n_pasi divizibil cu n_rebal.
    Intoarce eroarea de acoperire la scadenta (valoarea portofoliului de acoperire minus plata optiunii)."""
    npath, nstep = paths.shape[0], paths.shape[1] - 1
    every = nstep // n_rebal
    dt = T / nstep
    S0 = paths[:, 0]
    V0 = bs_price(S0, K, T, r, sigma_imp, kind)
    delta = bs_greeks(S0, K, T, r, sigma_imp, kind)['delta']
    cash = V0 - delta * S0
    for i in range(1, nstep + 1):
        cash = cash * np.exp(r * dt)
        if i % every == 0 and i < nstep:
            tau = T - i * dt
            d_new = bs_greeks(paths[:, i], K, tau, r, sigma_imp, kind)['delta']
            cash -= (d_new - delta) * paths[:, i]
            delta = d_new
    ST = paths[:, -1]
    payoff = np.maximum(ST - K, 0) if kind == 'call' else np.maximum(K - ST, 0)
    return cash + delta * ST - payoff

## Helper functions

In [ ]:
def btc_surface():
    """SVI pe fiecare scadenta a lantului BTC (optiuni OTM cotate, cel putin 2 zile pana la scadenta)."""
    c = deribit_chain()
    t0 = c['snapshot_utc'].iloc[0]
    c['T'] = (c['expiry'] - t0).dt.total_seconds() / (365 * 86400)
    c = c[(c['T'] >= 2 / 365) & (c['mark_iv'] > 0) & (c['bid'] > 0)]
    c = c[((c['type'] == 'put') & (c['strike'] < c['forward'])) | ((c['type'] == 'call') & (c['strike'] >= c['forward']))].copy()
    c['k'] = np.log(c['strike'] / c['forward'])
    c['w'] = (c['mark_iv'] / 100) ** 2 * c['T']
    c = c[c['k'].abs() < 1.2]
    fits = {}
    for e, g in c.groupby('expiry'):
        if len(g) < 7:
            continue
        f = svi_fit(g['k'].values, g['w'].values)
        T = g['T'].iloc[0]
        iv = lambda k: 100 * np.sqrt(svi_w(k, f['a'], f['b'], f['rho'], f['m'], f['s']) / T)
        fits[e] = dict(T=T, days=T * 365, F=float(g['forward'].iloc[0]), n=int(len(g)), atm=float(iv(0.0)),
                       rr=float(iv(0.15) - iv(-0.15)), wing_dn=float(iv(-0.3)), wing_up=float(iv(0.3)),
                       rmse_iv=float(np.sqrt(np.mean((iv(g['k'].values) - g['mark_iv'].values) ** 2))), **f)
    tab = pd.DataFrame(fits).T
    tab.index.name = 'expiry'
    tab.to_csv(os.path.join(HERE, 'ch12_btc_svi.csv'))
    return c, tab, t0


def pick(tab, days):
    """Scadenta cea mai apropiata de un numar dat de zile."""
    return (tab['days'] - days).abs().astype(float).idxmin()


def btc_vix(c_all=None):
    """Indice de tip VIX pe 30 de zile din lantul BTC (formula Cboe, r = 0, preturi mid in USD)."""
    c = deribit_chain() if c_all is None else c_all
    t0 = c['snapshot_utc'].iloc[0]
    c = c.copy()
    c['T'] = (c['expiry'] - t0).dt.total_seconds() / (365 * 86400)
    c = c[(c['bid'] > 0) & (c['ask'] > 0)]
    c['mid_usd'] = 0.5 * (c['bid'] + c['ask']) * c['forward']
    exps = sorted(c['expiry'].unique())
    Td = {e: c.loc[c['expiry'] == e, 'T'].iloc[0] * 365 for e in exps}
    near = max(e for e in exps if Td[e] < 30)
    nxt = min(e for e in exps if Td[e] >= 30)
    res = {}
    for tag, e in [('near', near), ('next', nxt)]:
        g = c[c['expiry'] == e]
        F = g['forward'].iloc[0]; T = g['T'].iloc[0]
        otm = pd.concat([g[(g['type'] == 'put') & (g['strike'] < F)], g[(g['type'] == 'call') & (g['strike'] >= F)]])
        v = variance_from_strip(otm['strike'].values, otm['mid_usd'].values, F, T)
        res[tag] = dict(expiry=str(pd.Timestamp(e).date()), days=float(T * 365), F=float(F), n=int(len(otm)), var=float(v),
                        vol=float(100 * np.sqrt(v)))
    T1, T2 = res['near']['days'], res['next']['days']
    w1 = (T2 - 30) / (T2 - T1)
    v30 = (T1 * res['near']['var'] * w1 + T2 * res['next']['var'] * (1 - w1)) / 30
    res['index'] = float(100 * np.sqrt(v30))
    return res


def vrp_sp500():
    """Prima de risc a variantei: VIX^2 minus varianta realizata in urmatoarele 21 de zile de tranzactionare."""
    px = pd.concat([load_close('sp500'), load_close('vix')], axis=1, join='inner').dropna()
    r = np.log(px['sp500']).diff()
    rv = 252 / 21 * (r ** 2).rolling(21).sum().shift(-21) * 1e4       # (%)^2 anualizat
    d = pd.DataFrame({'vix': px['vix'], 'iv2': px['vix'] ** 2, 'rv': rv, 'rv_past': rv.shift(21)}).dropna()
    d['vrp'] = d['iv2'] - d['rv']
    d['vrp_vol'] = d['vix'] - np.sqrt(d['rv'])
    return d

## Analysis

In [ ]:
def svi_gk(k, a, b, rho, m, s):
    """Functia g(k) a lui Gatheral-Jacquier (2014): fara arbitraj de tip fluture daca g(k) >= 0 si w > 0."""
    W = svi_w(k, a, b, rho, m, s)
    W1 = b * (rho + (k - m) / np.sqrt((k - m) ** 2 + s ** 2))
    W2 = b * s ** 2 / ((k - m) ** 2 + s ** 2) ** 1.5
    return (1 - k * W1 / (2 * W)) ** 2 - W1 ** 2 / 4 * (1 / W + 0.25) + W2 / 2


def svi_no_arbitrage(tab):
    """Verificarile de absenta a arbitrajului pentru fiecare SVI: panta aripilor (Lee, 2004), g(k) >= 0 si w(k) > 0
    (Gatheral-Jacquier, 2014), si arbitrajul de calendar dw/dtau >= 0 intre scadente consecutive."""
    kg = np.linspace(-1.5, 1.5, 3001)
    rows = {}
    for e, f in tab.iterrows():
        p = [float(f[x]) for x in ['a', 'b', 'rho', 'm', 's']]
        rows[str(pd.Timestamp(e).date())] = dict(days=float(f['days']), slope_left=p[1] * (1 - p[2]),
                                                 slope_right=p[1] * (1 + p[2]), wmin=float(svi_w(kg, *p).min()),
                                                 gmin=float(svi_gk(kg, *p).min()))
    ex = list(tab.index)
    kc = np.linspace(-0.5, 0.5, 201)
    cal = []
    for e1, e2 in zip(ex[:-1], ex[1:]):
        p1 = [float(tab.loc[e1, x]) for x in ['a', 'b', 'rho', 'm', 's']]
        p2 = [float(tab.loc[e2, x]) for x in ['a', 'b', 'rho', 'm', 's']]
        dw = svi_w(kc, *p2) - svi_w(kc, *p1)
        cal.append(dict(e1=str(pd.Timestamp(e1).date()), e2=str(pd.Timestamp(e2).date()), min_dw=float(dw.min()),
                        share_viol=float((dw < 0).mean())))
    lee = max(max(r['slope_left'], r['slope_right']) for r in rows.values())
    return dict(per_expiry=rows, calendar=cal, max_slope=float(lee), n_exp=len(rows),
                n_g_neg=int(sum(r['gmin'] < 0 for r in rows.values())),
                n_w_neg=int(sum(r['wmin'] <= 0 for r in rows.values())),
                n_cal_viol=int(sum(c['min_dw'] < 0 for c in cal)), n_pairs=len(cal))


def strip_variance(K, Q, F, T):
    """Integrala 2/T * int Q(K)/K^2 dK (trapez) pe o grila densa de preturi OTM (F = K0, r = 0)."""
    return 2 / T * integrate.trapezoid(Q / K ** 2, K)


def btc_vix_bias(tab, c_all=None):
    """Indicele de tip VIX pentru Bitcoin: efectul discretizarii si al trunchierii benzii de preturi de exercitare
    (Jiang-Tian, 2005), cu preturi SVI in locul cotatiilor; plus diferenta salt vs variatia patratica intr-un model Merton."""
    c = deribit_chain() if c_all is None else c_all
    t0 = c['snapshot_utc'].iloc[0]
    c = c.copy()
    c['T'] = (c['expiry'] - t0).dt.total_seconds() / (365 * 86400)
    c = c[(c['bid'] > 0) & (c['ask'] > 0)]
    base = btc_vix(c_all)
    res = {}
    for tag in ['near', 'next']:
        e = pd.Timestamp(base[tag]['expiry'] + ' 08:00:00')
        g = c[c['expiry'] == e]
        F = g['forward'].iloc[0]; T = g['T'].iloc[0]
        otm = pd.concat([g[(g['type'] == 'put') & (g['strike'] < F)], g[(g['type'] == 'call') & (g['strike'] >= F)]])
        f = tab.loc[e]
        p = [float(f[x]) for x in ['a', 'b', 'rho', 'm', 's']]
        def svi_q(K):
            sig = np.sqrt(np.clip(svi_w(np.log(K / F), *p), 1e-10, None) / T)
            return np.where(K < F, bs_price(F, K, T, 0.0, sig, 'put'), bs_price(F, K, T, 0.0, sig, 'call'))
        Kq = np.sort(otm['strike'].values)
        v_q = variance_from_strip(Kq, svi_q(Kq), F, T)                        # SVI la preturile cotate
        Kd = np.linspace(Kq.min(), Kq.max(), 20001)
        v_d = strip_variance(Kd, svi_q(Kd), F, T)                            # grila densa, acelasi interval
        Kw = F * np.exp(np.linspace(-4, 4, 40001))
        v_w = strip_variance(Kw, svi_q(Kw), F, T)                            # grila densa, interval extins
        res[tag] = dict(days=float(T * 365), quoted=base[tag]['var'], svi_quoted=float(v_q), svi_dense=float(v_d),
                        svi_wide=float(v_w), kf_min=float(Kq.min() / F), kf_max=float(Kq.max() / F))
    T1, T2 = res['near']['days'], res['next']['days']
    w1 = (T2 - 30) / (T2 - T1)
    idx = lambda key: float(100 * np.sqrt((T1 * res['near'][key] * w1 + T2 * res['next'][key] * (1 - w1)) / 30))
    out = dict(res, index_quoted=idx('quoted'), index_svi_quoted=idx('svi_quoted'), index_svi_dense=idx('svi_dense'),
               index_svi_wide=idx('svi_wide'))
    # salturi Merton (parametrii din graficul zambetelor): E^Q[-2 ln(S_T/F)] / T vs E^Q[QV] / T
    sigma, lam, mu_j, sig_j = 0.14, 1.0, -0.08, 0.08
    # sub masura de evaluare din merton_price: intensitate lam, salturi log ~ N(mu_j, sig_j^2)
    lq, mq = lam, mu_j
    EJ2 = mq ** 2 + sig_j ** 2
    EeJ = np.exp(mq + 0.5 * sig_j ** 2)
    qv = sigma ** 2 + lq * EJ2
    strip = sigma ** 2 + 2 * lq * (EeJ - 1 - mq)
    out['merton'] = dict(qv=float(qv), strip=float(strip), vol_qv=float(100 * np.sqrt(qv)), vol_strip=float(100 * np.sqrt(strip)),
                         rel=float(strip / qv - 1))
    return out


def qlike(rv, f):
    return rv / f - np.log(rv / f) - 1


def hac_se(u, lags):
    """Eroarea standard HAC (Newey-West, Bartlett) a mediei unei serii."""
    u = np.asarray(u, float)
    return nw_mean(u, lags)[1]


def iv_gmm(y, X, Z, lags):
    """Estimator IV exact identificat, b = (Z'X)^{-1} Z'y, cu varianta HAC (Newey-West) a momentelor Z u."""
    b = np.linalg.solve(Z.T @ X, Z.T @ y)
    u = y - X @ b
    g = Z * u[:, None]
    T = len(y)
    S = g.T @ g / T
    for L in range(1, lags + 1):
        G = g[L:].T @ g[:-L] / T
        S += (1 - L / (lags + 1)) * (G + G.T)
    A = np.linalg.inv(Z.T @ X / T)
    V = A @ S @ A.T / T
    return b, np.sqrt(np.diag(V))


def vix_forecast_inference():
    """VIX ca prognoza a variantei realizate pe 21 de zile: test comun Mincer-Zarnowitz, specificatia in logaritmi,
    erori in variabile (IV cu VIX^2 intarziat, Christensen-Prabhala), incluziune fata de HAR-RV (Corsi; Busch et al.)
    si comparatie in afara esantionului cu pierderea QLIKE si testul Diebold-Mariano (Patton, 2011)."""
    d = vrp_sp500().copy()
    out = {}
    L = 21
    m = sm.OLS(d['rv'], sm.add_constant(d[['iv2']])).fit(cov_type='HAC', cov_kwds={'maxlags': L})
    w = m.wald_test('const = 0, iv2 = 1', scalar=True)
    cv = m.cov_params()
    out['lev'] = dict(a=float(m.params['const']), b=float(m.params['iv2']), a_se=float(m.bse['const']), b_se=float(m.bse['iv2']),
                      corr_ab=float(cv.loc['const', 'iv2'] / np.sqrt(cv.loc['const', 'const'] * cv.loc['iv2', 'iv2'])),
                      wald=float(w.statistic), wald_p=float(w.pvalue), n=int(len(d)), r2=float(m.rsquared))
    px = pd.concat([load_close('sp500'), load_close('vix')], axis=1, join='inner').dropna()
    rd = (np.log(px['sp500']).diff() ** 2 * 252 * 1e4)
    d['har_d'] = rd.reindex(d.index)
    d['har_w'] = rd.rolling(5).mean().reindex(d.index)
    d['har_m'] = rd.rolling(22).mean().reindex(d.index)
    d['iv2_lag'] = d['iv2'].shift(21)
    d = d.dropna()
    ly, lx = np.log(d['rv']), np.log(d['iv2'])
    ml = sm.OLS(ly, sm.add_constant(lx.rename('liv2'))).fit(cov_type='HAC', cov_kwds={'maxlags': L})
    wl = ml.wald_test('liv2 = 1', scalar=True)
    out['log'] = dict(a=float(ml.params['const']), b=float(ml.params['liv2']), a_se=float(ml.bse['const']),
                      b_se=float(ml.bse['liv2']), r2=float(ml.rsquared), t_b1=float((ml.params['liv2'] - 1) / ml.bse['liv2']),
                      wald_b1=float(wl.statistic), wald_b1_p=float(wl.pvalue))
    X = np.column_stack([np.ones(len(d)), lx.values])
    Z = np.column_stack([np.ones(len(d)), np.log(d['iv2_lag']).values])
    b, se = iv_gmm(ly.values, X, Z, L)
    fs = sm.OLS(lx, sm.add_constant(np.log(d['iv2_lag']).rename('z'))).fit(cov_type='HAC', cov_kwds={'maxlags': L})
    out['iv'] = dict(a=float(b[0]), b=float(b[1]), a_se=float(se[0]), b_se=float(se[1]), t_b1=float((b[1] - 1) / se[1]),
                     first_stage_t=float(fs.tvalues['z']), first_stage_r2=float(fs.rsquared))
    # incluziune: log RV pe log VIX^2 si componentele HAR in logaritmi
    for c_ in ['har_d', 'har_w', 'har_m']:
        d['l' + c_] = np.log(d[c_].clip(lower=1e-2))
    me = sm.OLS(ly, sm.add_constant(pd.concat([lx.rename('liv2'), d[['lhar_d', 'lhar_w', 'lhar_m']]], axis=1))).fit(
        cov_type='HAC', cov_kwds={'maxlags': L})
    mh = sm.OLS(ly, sm.add_constant(d[['lhar_d', 'lhar_w', 'lhar_m']])).fit(cov_type='HAC', cov_kwds={'maxlags': L})
    wh = me.wald_test('lhar_d = 0, lhar_w = 0, lhar_m = 0', scalar=True)
    out['enc'] = dict(b_iv=float(me.params['liv2']), b_iv_se=float(me.bse['liv2']), r2=float(me.rsquared),
                      r2_har=float(mh.rsquared), r2_iv=float(ml.rsquared), wald_har=float(wh.statistic),
                      wald_har_p=float(wh.pvalue), b_m=float(me.params['lhar_m']), b_m_se=float(me.bse['lhar_m']))
    # in afara esantionului (din 2000): coeficienti reestimati la fiecare 21 de zile, doar cu tinte deja observate
    idx = d.index
    start = np.searchsorted(idx, pd.Timestamp('2000-01-03'))
    F_raw, F_mz, F_har, RV = [], [], [], []
    b_mz = b_har = None
    Xh_all = sm.add_constant(d[['lhar_d', 'lhar_w', 'lhar_m']]).values
    Xm_all = sm.add_constant(lx.rename('liv2')).values
    for i in range(start, len(d)):
        if (i - start) % 21 == 0:
            tr = slice(0, i - 21)                       # tintele RV_{s, s+21} cunoscute la data i: s <= i - 21
            b_har = np.linalg.lstsq(Xh_all[tr], ly.values[tr], rcond=None)[0]
            b_mz = np.linalg.lstsq(Xm_all[tr], ly.values[tr], rcond=None)[0]
            s2h = np.var(ly.values[tr] - Xh_all[tr] @ b_har)
            s2m = np.var(ly.values[tr] - Xm_all[tr] @ b_mz)
        F_raw.append(d['iv2'].values[i])
        F_mz.append(np.exp(Xm_all[i] @ b_mz + 0.5 * s2m))   # prognoza in nivel din regresia in logaritmi
        F_har.append(np.exp(Xh_all[i] @ b_har + 0.5 * s2h))
        RV.append(d['rv'].values[i])
    F_raw, F_mz, F_har, RV = map(np.array, (F_raw, F_mz, F_har, RV))
    oos = dict(start=str(idx[start].date()), n=int(len(RV)))
    for name, Fc in [('raw', F_raw), ('mz', F_mz)]:
        for lname, lf in [('qlike', qlike), ('mse', lambda y, f: (y - f) ** 2)]:
            dl = lf(RV, Fc) - lf(RV, F_har)          # < 0: VIX mai bun decat HAR
            se = hac_se(dl, L)
            oos[f'{name}_{lname}'] = dict(loss=float(lf(RV, Fc).mean()), loss_har=float(lf(RV, F_har).mean()),
                                          ratio=float(lf(RV, Fc).mean() / lf(RV, F_har).mean()), dm=float(dl.mean() / se),
                                          p=float(2 * (1 - stats.norm.cdf(abs(dl.mean() / se)))))
    out['oos'] = oos
    return out


def vrp_monthly():
    """Date lunare BTZ (2009): IV = VIX^2/12 la sfarsitul lunii, RV = suma randamentelor zilnice (%) la patrat din luna,
    VRP = IV - RV (ex ante), randamentul in exces = randamentul log lunar S&P 500 minus TB3MS/12 (FRED)."""
    spx, vix = read_market('GSPC.INDX')['close'], read_market('VIX.INDX')['close']
    px = pd.concat([spx, vix], axis=1, keys=['spx', 'vix']).dropna()
    r = 100 * np.log(px['spx']).diff().dropna()
    rv = (r ** 2).resample('ME').sum()
    iv = (px['vix'] ** 2 / 12).resample('ME').last()
    rm = 100 * np.log(px['spx'].resample('ME').last()).diff()
    tb = pd.read_csv('https://fred.stlouisfed.org/graph/fredgraph.csv?id=TB3MS', index_col=0, parse_dates=True).iloc[:, 0]
    tb.index = tb.index + pd.offsets.MonthEnd(0)
    df = pd.DataFrame({'iv': iv, 'rv': rv, 'rm': rm, 'rf': tb / 12}).dropna()
    df = df[df.index >= '1990-01-31']
    df['vrp'] = df['iv'] - df['rv']
    df['ex'] = df['rm'] - df['rf']
    return df


def hodrick_t(y, X, e1, h):
    """Erori standard Hodrick (1992) 1B pentru regresia cu randamente suprapuse (suma regresorilor trecuti)."""
    T = len(y)
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    Z = X.T @ X / T
    S = np.zeros((X.shape[1], X.shape[1]))
    for t in range(h - 1, T):
        w = e1[t] * X[t - h + 1:t + 1].sum(axis=0)
        S += np.outer(w, w)
    S /= T
    Zi = np.linalg.inv(Z)
    V = Zi @ S @ Zi / T
    return b, b / np.sqrt(np.diag(V))


def vrp_predict(B=2000):
    """Predictibilitatea randamentelor prin VRP ex ante (BTZ, 2009): regresii lunare suprapuse pe h = 1, 3, 6, 12 luni,
    cu erori standard Newey-West (h lag-uri), Hansen-Hodrick (h - 1 lag-uri, nucleu uniform) si Hodrick (1992) 1B;
    p-valoare bootstrap sub H0 (VRP ca AR(1), reziduuri reesantionate impreuna; Stambaugh, 1999)."""
    df = vrp_monthly()
    ex, v = df['ex'].values, df['vrp'].values
    n = len(ex)

    def stats_h(ex, v, h):
        y = np.array([(12 / h) * ex[t + 1:t + 1 + h].sum() if t + h < n else np.nan for t in range(n)])
        ok = ~np.isnan(y)
        yy, vv = y[ok], v[ok]
        X = np.column_stack([np.ones(len(yy)), vv])
        e1 = (12 / h) * (np.append(ex[1:], np.nan)[ok] - np.nanmean(ex[1:]))
        b, th = hodrick_t(yy, X, np.nan_to_num(e1), h)
        u = yy - X @ b
        T = len(yy)
        g = X * u[:, None]
        def lrv(lags, kern):
            S = g.T @ g / T
            for L_ in range(1, lags + 1):
                G = g[L_:].T @ g[:-L_] / T
                S += kern(L_, lags) * (G + G.T)
            A = np.linalg.inv(X.T @ X / T)
            return A @ S @ A / T
        Vnw = lrv(h, lambda L_, m: 1 - L_ / (m + 1))
        Vhh = lrv(h - 1, lambda L_, m: 1.0)
        se_hh = np.sqrt(Vhh[1, 1]) if Vhh[1, 1] > 0 else np.nan
        r2 = 1 - u.var() / yy.var()
        return dict(b=float(b[1]), t_nw=float(b[1] / np.sqrt(Vnw[1, 1])), t_hh=float(b[1] / se_hh),
                    t_hod=float(th[1]), r2=float(100 * r2), n=int(T))
    out = {}
    # AR(1) pentru VRP si randamente sub H0
    phi = np.polyfit(v[:-1], v[1:], 1)
    ev = v[1:] - np.polyval(phi, v[:-1])
    er = ex[1:] - ex[1:].mean()
    rng = np.random.default_rng(SEED)
    for h in [1, 3, 6, 12]:
        out[str(h)] = stats_h(ex, v, h)
    tb = {h: [] for h in [1, 3, 6, 12]}
    for _ in range(B):
        j = rng.integers(0, len(ev), len(ev))
        vb = np.empty(n); vb[0] = v[rng.integers(0, n)]
        for t in range(1, n):
            vb[t] = phi[1] + phi[0] * vb[t - 1] + ev[j[t - 1]]
        xb = np.empty(n); xb[0] = ex.mean(); xb[1:] = ex[1:].mean() + er[j]
        for h in [1, 3, 6, 12]:
            tb[h].append(stats_h(xb, vb, h)['t_hod'])
    for h in [1, 3, 6, 12]:
        a = np.array(tb[h])
        out[str(h)]['p_boot'] = float(np.mean(a >= out[str(h)]['t_hod']))
        out[str(h)]['t_boot95'] = float(np.quantile(a, 0.95))
    out['phi'] = float(phi[0]); out['start'] = df.index[0].strftime('%Y-%m'); out['end'] = df.index[-1].strftime('%Y-%m')
    return out

## Run

In [ ]:
c, tab, t0 = btc_surface()
print(svi_no_arbitrage(tab))
print(btc_vix_bias(tab))
print(vix_forecast_inference())
vrp_predict()